# 01 · Exploratory data analysis

What the corpus looks like, and the three properties that shape every later
decision: the labels are **rare**, they are **correlated**, and they are
**hierarchical** (`severe_toxic` never occurs without `toxic`).

In [ ]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

from src.config import Config

# `memory` keeps this session's intermediate frames out of the file system;
# switch to "file" to persist them under data/processed/.
CONFIG = Config(persistence_backend="memory")
CONFIG.ensure_directories()
CONFIG.label_columns

In [ ]:
from src.dataio import load_training_data, raw_files_available

# allow_synthetic keeps the notebook runnable before the Kaggle download;
# every number below is only meaningful once raw_files_available() is True.
train = load_training_data(config=CONFIG, allow_synthetic=True)
print("real dataset:", raw_files_available(), "| rows:", len(train))
train.head(3)

## Label frequency

The first number to internalise: how little positive signal there is.

In [ ]:
labels = list(CONFIG.label_columns)
counts = train[labels].sum().sort_values(ascending=False)
summary = pd.DataFrame({"count": counts, "rate": counts / len(train)})
display(summary.style.format({"rate": "{:.4%}"}))

fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(x=counts.values, y=counts.index, ax=ax, color="#4C72B0")
ax.set(xlabel="positive comments", ylabel="", title="Label frequency")
for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3, fontsize=9)
sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(CONFIG.figures_dir / "label_frequency.png", dpi=200)

In [ ]:
flagged = train[labels].sum(axis=1) > 0
print(f"clean comments: {(~flagged).mean():.2%}")
print(f"flagged comments: {flagged.mean():.2%}")
print("\nlabels per flagged comment:")
print(train.loc[flagged, labels].sum(axis=1).value_counts().sort_index())

## Label co-occurrence

Binary relevance trains the six labels independently. The matrix below shows
what that assumption throws away — quantify it here so the report can argue the
point rather than assert it.

In [ ]:
cooccurrence = train[labels].T.dot(train[labels])
conditional = cooccurrence.div(np.diag(cooccurrence), axis=0)  # P(column | row)

fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(conditional, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
            cbar_kws={"label": "P(column | row)"}, ax=ax)
ax.set_title("Conditional label co-occurrence")
fig.tight_layout()
fig.savefig(CONFIG.figures_dir / "label_cooccurrence.png", dpi=200)

# P(toxic | severe_toxic) == 1.0 is the hierarchy: severe_toxic is a subset.
print("P(toxic | severe_toxic) =", conditional.loc["severe_toxic", "toxic"])

## Comment length

Drives the `max_length` of the transformer: truncation loses information.

In [ ]:
train["n_chars"] = train[CONFIG.text_column].str.len()
train["n_words"] = train[CONFIG.text_column].str.split().str.len()

print(train[["n_chars", "n_words"]].describe(percentiles=[.5, .9, .95, .99]).round(1))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
sns.histplot(train["n_words"].clip(upper=400), bins=60, ax=axes[0], color="#4C72B0")
axes[0].set(title="Comment length (words, clipped at 400)", xlabel="words")

length_by_label = pd.DataFrame({
    "toxic": train.loc[train.toxic == 1, "n_words"].clip(upper=400),
    "clean": train.loc[train.toxic == 0, "n_words"].clip(upper=400),
})
sns.boxplot(data=length_by_label, ax=axes[1], showfliers=False)
axes[1].set(title="Length by toxicity", ylabel="words")
sns.despine(fig=fig)
fig.tight_layout()
fig.savefig(CONFIG.figures_dir / "comment_length.png", dpi=200)

## What cleaning does

Inspect the transformation before trusting it.

In [ ]:
from src.preprocessing import TextCleaner

cleaner = TextCleaner()
examples = train[CONFIG.text_column].head(5).tolist()
for raw in examples:
    print("RAW  :", raw[:110].replace("\n", " "))
    print("CLEAN:", cleaner.clean(raw)[:110])
    print("-" * 100)

## Takeaways for the modelling stage

1. ~90% of the comments carry no label → accuracy is uninformative; report AUC,
   PR-AUC and per-label F1 instead.
2. `threat` and `identity_hate` are one or two orders of magnitude rarer than
   `toxic` → folds must be stratified over all labels simultaneously.
3. Labels are strongly correlated and partly hierarchical → binary relevance is
   a simplification whose cost should be measured, not assumed away.
4. The length distribution is heavily right-skewed → a fixed truncation length
   is cheap for the transformer; check the percentile it covers.